# Predicting Victim Counts in U.S. Mass Shootings Using Machine Learning Models

This notebook builds and compares several regression models to predict the total number of victims in U.S. mass shootings using selected categorical features, including race, gender, and reported mental health status.

The goal of this project is to practice data cleaning, feature encoding, model training, model evaluation, and basic feature importance analysis.

### Importing Libraries

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import plotly as py
from plotly.offline import iplot, init_notebook_mode
import plotly.figure_factory as ff
import matplotlib.pyplot as plt
import plotly.express as px
%matplotlib inline

### Load Dataset

In [60]:
df = pd.read_csv("data/mass_shootings_dataset.csv", encoding="latin-1")
df.head()

,S#,Title,Location,Date,Summary,Fatalities,Injured,Total victims,Mental Health Issues,Race,Gender,Latitude,Longitude
0,1,Las Vegas Strip mass shooting,"Las Vegas, NV",10/1/2017,NaN,58,515,573,Unclear,NaN,NaN,NaN,NaN
1,2,San Francisco UPS shooting,"San Francisco, CA",6/14/2017,"Jimmy Lam, 38, fatally shot three coworkers an...",3,2,5,Yes,Asian,M,NaN,NaN
2,3,Pennsylvania supermarket shooting,"Tunkhannock, PA",6/7/2017,"Randy Stair, a 24-year-old worker at Weis groc...",3,0,3,Unclear,White,M,NaN,NaN
3,4,Florida awning manufacturer shooting,"Orlando, Florida",6/5/2017,"John Robert Neumann, Jr., 45, a former employe...",5,0,5,Unclear,NaN,M,NaN,NaN
4,5,Rural Ohio nursing home shooting,"Kirkersville, Ohio",5/12/2017,"Thomas Hartless, 43, shot and killed a former ...",3,0,3,Yes,White,M,NaN,NaN


### Data Cleaning

The original dataset contains several columns that are not needed for this modeling notebook. I kept the variables used for prediction and removed identifying or descriptive columns.

In [61]:
# Drop columns that are not used in the modeling process

df = df.drop(["S#", "Title", "Summary", "Latitude", "Longitude", "Fatalities", "Injured", "Location", "Date"], axis=1)
df.head()

,Total victims,Mental Health Issues,Race,Gender
0,573,Unclear,NaN,NaN
1,5,Yes,Asian,M
2,3,Unclear,White,M
3,5,Unclear,NaN,M
4,3,Yes,White,M


In [62]:
# Standardize inconsistent category labels

df.loc[df["Gender"]== "M", "Gender" ]="Male"

df.loc[df["Gender"]== "F", "Gender" ]="Female"

df.loc[df["Gender"]== "M/F", "Gender" ]="Male/Female"

df.loc[df["Gender"]== "Male/Female", "Gender" ]="Unknown"

df.loc[df["Race"]== "Black American or African American", "Race" ]="Black"

df.loc[df["Race"]== "Black American or African American/Unknown", "Race" ]="Black"

df.loc[df["Race"]== "Asian American", "Race" ]="Asian"

df.loc[df["Race"]== "Asian American/Some other race", "Race" ]="Asian"

df.loc[df["Race"]== "White American or European American", "Race" ]="White"

df.loc[df["Race"]== "black", "Race" ]="Black"

df.loc[df["Race"]== "White ", "Race" ]="White"

df.loc[df["Race"]== "Native American or Alaska Native", "Race" ]="Native American"

df.loc[df["Race"]== "White American or European American/Some other Race", "Race" ]="White"

df.loc[df["Race"]== "Some other race", "Race" ]="Other"

df.loc[df["Race"]== "Two or more races", "Race" ]="Other"

df.loc[df["Race"]== "unclear", "Race" ]="Unknown"

df.loc[df["Race"]== "white", "Race" ]="White"

df["Race"] = df["Race"].fillna("Unknown")
df["Gender"] = df["Gender"].fillna("Unknown")
df["Mental Health Issues"] = df["Mental Health Issues"].fillna("Unknown")

df.loc[df["Mental Health Issues"] == "unknown", "Mental Health Issues" ]="Unknown"

df.loc[df["Mental Health Issues"] == "Unclear", "Mental Health Issues" ]="Unknown"

df.loc[df["Mental Health Issues"] == "Unclear ", "Mental Health Issues" ]="Unknown"

### Feature Selection

For this model, I used race, gender, and reported mental health status as categorical predictors. The target variable is total victims.

In [63]:
# Select categorical predictor variables

X = df[["Mental Health Issues", "Race", "Gender"]]

In [64]:
# Convert categorical variables into numeric dummy variables

cat_numerical = pd.get_dummies(X, drop_first=True)
cat_numerical.head()

,Mental Health Issues_Unknown,Mental Health Issues_Yes,Race_Black,Race_Latino,Race_Native American,Race_Other,Race_Unknown,Race_White,Gender_Male,Gender_Unknown
0,True,False,False,False,False,False,True,False,False,True
1,False,True,False,False,False,False,False,False,True,False
2,True,False,False,False,False,False,False,True,True,False
3,True,False,False,False,False,False,True,False,True,False
4,False,True,False,False,False,False,False,True,True,False


In [65]:
# Combine the target variable with the encoded predictor variables

model_df = pd.concat([df[["Total victims"]], cat_numerical], axis=1)
model_df.head()

,Total victims,Mental Health Issues_Unknown,Mental Health Issues_Yes,Race_Black,Race_Latino,Race_Native American,Race_Other,Race_Unknown,Race_White,Gender_Male,Gender_Unknown
0,573,True,False,False,False,False,False,True,False,False,True
1,5,False,True,False,False,False,False,False,False,True,False
2,3,True,False,False,False,False,False,False,True,True,False
3,5,True,False,False,False,False,False,True,False,True,False
4,3,False,True,False,False,False,False,False,True,True,False


## Train-Test Split

The dataset is split into training and testing sets. The feature variables are scaled so that models such as Random Forest, LASSO, and Support Vector Regression perform more reliably.

In [66]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target variable
X = model_df.drop("Total victims", axis=1)
y = model_df["Total victims"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0
)

# Scale the feature variables
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Linear Regression Test

In [67]:
from sklearn.linear_model import LinearRegression

# Train the Linear Regression Model
lin_reg = LinearRegression()
regressor = lin_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = regressor.predict(X_test)

from sklearn import metrics

# Evaluate the model
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Mean Absolute Error: 8.987760513405423
Mean Squared Error: 258.0700945781542
Root Mean Squared Error: 16.064560204940385


### K-Nearest Neighbor Test

In [68]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(random_state=42, n_estimators=500)
regressor = rf_reg.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

from sklearn import metrics

print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Mean Absolute Error: 8.44316645349424
Mean Squared Error: 229.31321843083379
Root Mean Squared Error: 15.14309144233217


### Support Vector Machine (Regressor) Test

In [69]:
from sklearn import svm
svm_reg = svm.SVR()

regressor = svm_reg.fit(X_train, y_train)
y_pred = regressor.predict(X_test)


from sklearn import metrics

print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Mean Absolute Error: 5.28878853942417
Mean Squared Error: 166.39382961542384
Root Mean Squared Error: 12.899373225681309


### Cross-Validation

Cross-validation provides a more stable estimate of model performance by evaluating the model across multiple train-test splits.

In [70]:
from sklearn.model_selection import cross_val_score

print(cross_val_score(regressor, X, y, cv=5, scoring ="neg_mean_absolute_error"))

[-10.54339667  -2.96306719  -6.22102287  -5.61584449  -7.47443058]


### LASSO Regression Model

In [71]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)  # Adjust the alpha parameter as needed
lasso.fit(X_train, y_train)

# Predict on the test set
y_pred = lasso.predict(X_test)

# Evaluate the model (e.g., using mean squared error)
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Mean Absolute Error: 8.835750995592415
Mean Squared Error: 253.0744189294297
Root Mean Squared Error: 15.908312887588982


### Random Forest Regression Model

In [72]:
from sklearn.ensemble import RandomForestRegressor
rf_reg = RandomForestRegressor(random_state=42, n_estimators=50)
regressor = rf_reg.fit(X_train, y_train)
y_pred = regressor.predict(X_test)


from sklearn import metrics

print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Mean Absolute Error: 8.456682258998418
Mean Squared Error: 237.91191690058653
Root Mean Squared Error: 15.42439356670422


### Permutation Importance

In [73]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance for the Random Forest model
result = permutation_importance(
    regressor,
    X_test,
    y_test,
    n_repeats=30,
    random_state=0
)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": result.importances_mean,
    "Standard Deviation": result.importances_std
}).sort_values(by="Importance", ascending=False)

importance_df

,Feature,Importance,Standard Deviation
1,Mental Health Issues_Yes,0.103058,0.044369
5,Race_Other,0.028643,0.022940
3,Race_Latino,0.003054,0.005469
4,Race_Native American,0.001294,0.001658
8,Gender_Male,0.000940,0.018591
2,Race_Black,-0.001907,0.016841
7,Race_White,-0.004876,0.018138
0,Mental Health Issues_Unknown,-0.033844,0.037043
6,Race_Unknown,-0.164003,0.057595
9,Gender_Unknown,-0.325225,0.063367


## Conclusion

This notebook compared multiple regression models for predicting total victim counts in U.S. mass shootings using race, gender, and reported mental health status as categorical predictors.

The models were evaluated using MAE, MSE, and RMSE. Because the features are limited and the topic is sensitive, the results should not be interpreted as causal explanations. Instead, this project demonstrates data cleaning, categorical encoding, model training, model evaluation, and feature importance analysis.

Future improvements could include adding more predictors such as year, state, location type, weapon type, and other contextual variables.